
CSYE7105 High Performance Parallel Machine Learning and AI

Instructor: Dr. Handan Liu

Lecture 11: Dask 1 - Arrays in Parallel


In [3]:
import numpy as np
import dask.array as da

### Blocked Algorithms in a nutshell
Let's do side by side the sum of the elements of an array using a NumPy array and a Dask array.

In [5]:
# NumPy array
x = np.ones(10)
x

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [6]:
%timeit x.sum()

508 ns ± 3.71 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [7]:
%timeit x[:5].sum() + x[5:].sum()

1.19 μs ± 13.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [8]:
x.sum()

10.0

In [9]:
x_da = da.ones(10, chunks=5)
x_da

dask.array<ones_like, shape=(10,), dtype=float64, chunksize=(5,), chunktype=numpy.ndarray>

In [10]:
y = x_da.sum()
y

dask.array<sum-aggregate, shape=(), dtype=float64, chunksize=(), chunktype=numpy.ndarray>

In [19]:
y.compute()

NameError: name 'y' is not defined

In [17]:
y.visualize(engine="cytoscape")

NameError: name 'y' is not defined

In [ ]:
graphviz

In [ ]:
y.compute()

### Task Graphs

In [ ]:
import dask.array as da
x = da.random.normal(size=500, chunks=100)
x

In [ ]:
x.visualize()

In [ ]:
x.mean().compute()


## Visualize the low level graph


In [ ]:
import dask.array as da
x = da.ones((15, 15), chunks=(5, 5))
x

In [ ]:
y = x + x.T
y.compute()

In [ ]:
# visualize the low level Dask graph
y.visualize(filename='transpose.svg', rankdir='LR')

In [ ]:
# visualize the low level Dask graph using cytoscape
y.visualize(engine="cytoscape")

In [ ]:
import dask
with dask.config.set({"visualization.engine": "cytoscape"}):
    y.visualize()


## Visualize the high level graph


In [ ]:
import dask.array as da
x = da.ones((15, 15), chunks=(5, 5))
y = x + x.T

# visualize the high level Dask graph
y.dask.visualize(filename='transpose-hlg.svg')

In [ ]:
y.dask  

### Create Dask Arrays

In [ ]:
da.from_array?

### Create Random array

In [42]:
darr = da.random.random((1000, 1000, 1000))
darr

NameError: name 'da' is not defined

In [ ]:
import dask.array as da
x = da.random.random((10000, 10000), chunks=(1000, 1000))
x

In [ ]:
y = x + x.T
z = y[::2, 5000:].mean(axis=1)
z

In [ ]:
zz = z.compute()
zz.shape

In [ ]:
%%timeit 
y = x + x.T
z = y[::2, 5000:].mean(axis=1)
z.compute()

In [ ]:
xnp = np.random.random((10000, 10000))
xnp.shape

In [ ]:
%%timeit
yy = xnp + xnp.T
zz = yy[::2, 5000:].mean(axis=1)
zz

### Performance comparison
Large Array

#### NumPy

In [ ]:
%%time
xn = np.random.normal(10, 0.1, size=(30_000, 30_000))
yn = xn.mean(axis=0)
yn

#### Dask

In [ ]:
%%time
xd = da.random.normal(10, 0.1, size=(30_000, 30_000), chunks=(3000, 3000))
yd = xd.mean(axis=0)
yd.compute()

In [ ]:
%%time
xd = da.random.normal(10, 0.1, size=(30_000, 30_000), chunks=(10000, 10000))
yd = xd.mean(axis=0)
yd.compute()

Let's see different chunk size

In [ ]:
x2 = da.random.random((10000, 10000), chunks=(500, 500))
x2

In [ ]:
%%timeit 
y2 = x2 + x2.T
z2 = y2[::2, 5000:].mean(axis=1)
z2.compute()

no longer square-like

In [ ]:
x3 = da.random.random((10000, 10000), chunks=(1000, 500))
x3

In [ ]:
%%timeit 
y3 = x3 + x3.T
z3 = y3[::2, 5000:].mean(axis=1)
z3.compute()


## Unknow Chunks


In [ ]:
x = da.from_array(np.random.randn(100), chunks=20)
x

In [ ]:
x += 0.1
x

In [ ]:
x[4]

In [ ]:
y = x[x > 0]

In [ ]:
y

In [ ]:
y.shape

In [ ]:
y[4]

In [ ]:
y.compute_chunk_sizes()

In [ ]:
y.compute().shape

In [ ]:
y[4].compute()


### Rechunk


In [ ]:
x = da.random.random((10000, 10000), chunks=(1000, 1000))
x

In [ ]:
x = x.rechunk((500, 1000))
x

In [ ]:
x = x.rechunk({0: 50, 1: 1000})
x